In [1]:
import pyspark
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/21 10:32:30 WARN Utils: Your hostname, codespaces-886ad4, resolves to a loopback address: 127.0.0.1; using 10.0.0.63 instead (on interface eth0)
26/04/21 10:32:30 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/21 10:32:36 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
import pandas as pd
from pyspark.sql import types

In [4]:
pd_green = pd.read_csv("/tmp/green/green_tripdata_2020-01.csv.gz", nrows=1000)

In [5]:
spark.createDataFrame(pd_green).schema

StructType([StructField('VendorID', LongType(), True), StructField('lpep_pickup_datetime', StringType(), True), StructField('lpep_dropoff_datetime', StringType(), True), StructField('store_and_fwd_flag', StringType(), True), StructField('RatecodeID', LongType(), True), StructField('PULocationID', LongType(), True), StructField('DOLocationID', LongType(), True), StructField('passenger_count', LongType(), True), StructField('trip_distance', DoubleType(), True), StructField('fare_amount', DoubleType(), True), StructField('extra', DoubleType(), True), StructField('mta_tax', DoubleType(), True), StructField('tip_amount', DoubleType(), True), StructField('tolls_amount', DoubleType(), True), StructField('ehail_fee', DoubleType(), True), StructField('improvement_surcharge', DoubleType(), True), StructField('total_amount', DoubleType(), True), StructField('payment_type', LongType(), True), StructField('trip_type', LongType(), True), StructField('congestion_surcharge', DoubleType(), True)])

In [6]:
green_schema = types.StructType([
     types.StructField('VendorID',  types.IntegerType(), True), 
     types.StructField('lpep_pickup_datetime',  types.TimestampType(), True), 
     types.StructField('lpep_dropoff_datetime',  types.TimestampType(), True), 
     types.StructField('store_and_fwd_flag',  types.StringType(), True), 
     types.StructField('RatecodeID',  types.IntegerType(), True), 
     types.StructField('PULocationID',  types.IntegerType(), True), 
     types.StructField('DOLocationID',  types.IntegerType(), True), 
     types.StructField('passenger_count',  types.IntegerType(), True), 
     types.StructField('trip_distance',  types.DecimalType(), True), 
     types.StructField('fare_amount',  types.DecimalType(), True), 
     types.StructField('extra',  types.DecimalType(), True), 
     types.StructField('mta_tax',  types.DecimalType(), True), 
     types.StructField('tip_amount',  types.DecimalType(), True), 
     types.StructField('tolls_amount',  types.DecimalType(), True), 
     types.StructField('ehail_fee',  types.DecimalType(), True), 
     types.StructField('improvement_surcharge',  types.DecimalType(), True), 
     types.StructField('total_amount',  types.DecimalType(), True), 
     types.StructField('payment_type',  types.IntegerType(), True), 
     types.StructField('trip_type',  types.IntegerType(), True), 
     types.StructField('congestion_surcharge',  types.DecimalType(), True)
])

In [7]:
pd_yellow = pd.read_csv("/tmp/yellow/yellow_tripdata_2020-01.csv.gz", nrows=1000)
spark.createDataFrame(pd_yellow).schema


StructType([StructField('VendorID', LongType(), True), StructField('tpep_pickup_datetime', StringType(), True), StructField('tpep_dropoff_datetime', StringType(), True), StructField('passenger_count', LongType(), True), StructField('trip_distance', DoubleType(), True), StructField('RatecodeID', LongType(), True), StructField('store_and_fwd_flag', StringType(), True), StructField('PULocationID', LongType(), True), StructField('DOLocationID', LongType(), True), StructField('payment_type', LongType(), True), StructField('fare_amount', DoubleType(), True), StructField('extra', DoubleType(), True), StructField('mta_tax', DoubleType(), True), StructField('tip_amount', DoubleType(), True), StructField('tolls_amount', DoubleType(), True), StructField('improvement_surcharge', DoubleType(), True), StructField('total_amount', DoubleType(), True), StructField('congestion_surcharge', DoubleType(), True)])

In [8]:
yellow_schema = types.StructType([
    types.StructField('VendorID', types.IntegerType(), True), 
    types.StructField('tpep_pickup_datetime', types.StringType(), True), 
    types.StructField('tpep_dropoff_datetime', types.StringType(), True), 
    types.StructField('passenger_count', types.IntegerType(), True), 
    types.StructField('trip_distance', types.DecimalType(), True), 
    types.StructField('RatecodeID', types.IntegerType(), True), 
    types.StructField('store_and_fwd_flag', types.StringType(), True), 
    types.StructField('PULocationID', types.IntegerType(), True), 
    types.StructField('DOLocationID', types.IntegerType(), True), 
    types.StructField('payment_type', types.IntegerType(), True), 
    types.StructField('fare_amount', types.DecimalType(), True), 
    types.StructField('extra', types.DecimalType(), True), 
    types.StructField('mta_tax', types.DecimalType(), True), 
    types.StructField('tip_amount', types.DecimalType(), True), 
    types.StructField('tolls_amount', types.DecimalType(), True), 
    types.StructField('improvement_surcharge', types.DecimalType(), True), 
    types.StructField('total_amount', types.DecimalType(), True), 
    types.StructField('congestion_surcharge', types.DecimalType(), True)])

In [ ]:
# partition - количество партиций (в данном случае паркет файлов) на каждый gz файл. Т.е. по 4 паркет файла на gz файл
for type in ["yellow", "green"]:
    for i in range(1,13):
        file = f"/tmp/{type}/{type}_tripdata_2020-{i:02d}.csv.gz"
        output = f"/tmp/{type}/pq/"
        schema = green_schema if type == "green" else yellow_schema
        df = spark.read \
            .option("header", "true") \
            .schema(schema) \
            .csv(file)
        df.repartition(4).write.parquet(output, mode='append')

In [17]:
df_green = spark.read.parquet("/tmp/green/pq")

In [19]:
df_yellow = spark.read.parquet("/tmp/yellow/pq")

In [ ]:
# приводим к общему виду для возможности объединять через Union
df_green = df_green \
    .withColumnRenamed('lpep_pickup_datetime', 'pickup_datetime') \
    .withColumnRenamed('lpep_dropoff_datetime', 'dropoff_datetime')

df_yellow = df_yellow  \
    .withColumnRenamed('tpep_pickup_datetime', 'pickup_datetime') \
    .withColumnRenamed('tpep_dropoff_datetime', 'dropoff_datetime')



In [ ]:
# находим общие колонки, порядок оставляем такой же, как в файле yellow (для удобства). 
# set(green.columns) & set(yellow.columns2) тоже даст пересечение, но в рандомном порядке. Причем, каждое выполнение кода может давать новый порядок.
common_cols = []
for col1 in df_yellow.columns:
    common_cols.append(col1) if col1 in df_green.columns else None
        

In [37]:
from pyspark.sql import functions as F

In [ ]:
# str - обращение к колонке, F.lit(str) - значение. Просто 'green' будет считаться обращением к колонке 'green'
# with column создает колонку
df_green_selected = df_green \
    .select(common_cols) \
    .withColumn('service_type', F.lit('green'))

df_yellow_selected = df_yellow \
    .select(common_cols) \
    .withColumn('service_type', F.lit('yellow'))

In [39]:
df_trips = df_green_selected.unionAll(df_yellow_selected)

In [45]:
df_trips.groupBy('service_type').count().show()

+------------+--------+
|service_type|   count|
+------------+--------+
|       green| 1734051|
|      yellow|24648499|
+------------+--------+



In [ ]:
# we need to create a table so we can address the data in FROM while quering with SQL
df_trips.registerTempTable('trips_data')

/workspaces/Data-Engineering/.venv/lib/python3.13/site-packages/pyspark/sql/classic/dataframe.py:178: FutureWarning: Deprecated in 2.0, use createOrReplaceTempView instead.
  warnings.warn("Deprecated in 2.0, use createOrReplaceTempView instead.", FutureWarning)


In [49]:
df_result = spark.sql("""
SELECT 
    -- Revenue grouping 
    PULocationID AS revenue_zone,
    date_trunc('month', pickup_datetime) AS revenue_month, 
    service_type, 

    -- Revenue calculation 
    SUM(fare_amount) AS revenue_monthly_fare,
    SUM(extra) AS revenue_monthly_extra,
    SUM(mta_tax) AS revenue_monthly_mta_tax,
    SUM(tip_amount) AS revenue_monthly_tip_amount,
    SUM(tolls_amount) AS revenue_monthly_tolls_amount,
    SUM(improvement_surcharge) AS revenue_monthly_improvement_surcharge,
    SUM(total_amount) AS revenue_monthly_total_amount,
    SUM(congestion_surcharge) AS revenue_monthly_congestion_surcharge,

    -- Additional calculations
    AVG(passenger_count) AS avg_monthly_passenger_count,
    AVG(trip_distance) AS avg_monthly_trip_distance
FROM
    trips_data
GROUP BY
    1, 2, 3
""")

In [ ]:
# coalesce(1) - reduce the number of partitions. so it will be 1 partition (and 1 parquet file) in output
df_result.coalesce(1).write.parquet('data/report/', mode='overwrite')

In [51]:
spark.stop()